# 개정 크롤링
### 조건 별 크롤링
- 불러오는 데이터 : 1차 크롤링을 바탕으로 확인한 데이터들
- 검색했을 때 첫번째 나오는 것들인 경우는 이전 방법대로 정상적으로 진행
- 두번째 이후 나온 것들은 나온 것들끼리 묶어서 진행
    - 포맷팅 이용해서 크롤링해보기
- 검색했는데 나오지 않는 것들은 제외
- 예상 한 300개 정도 생략될 것임 -> 나중에 추가로 찾을 수 있다면 찾아보는 것으로?
- 이후에 이를 중심으로 리뷰를 분석해보고, 협업 필터링을 중심으로 추천시스템 구현

In [1]:
import pandas as pd
import numpy as np
from selenium import webdriver
import time
from bs4 import BeautifulSoup
import requests
from selenium.webdriver.common.keys import Keys
import urllib

In [2]:
# 처리 완료되면 바꿔줘야함
df=pd.read_csv('./data/죽기전에 가봐야할 101가지 여행지.csv')
df

,여행지
0,경복궁
1,삼청동 거리
2,인사동
3,조계사
4,청와대
...,...
994,쇠소깍
995,백조일손지묘
996,휴애리 자연생활공원
997,안덕계곡


In [3]:
df=pd.read_csv('./data/여행지 명칭 확인_통합본.csv')
df

,여행지,일치여부,다른 칸에 존재여부
0,경복궁,O,NaN
1,삼청동 거리,O,NaN
2,인사동,O,NaN
3,조계사,O,NaN
4,청와대,O,NaN
...,...,...,...
999,쇠소깍,O,NaN
1000,백조일손지묘,X,X
1001,휴애리 자연생활공원,O,NaN
1002,안덕계곡,X,X


In [4]:
df_c=df[df['일치여부']=='O']
df_c=df_c[['여행지','일치여부']]
df_c.head()

,여행지,일치여부
0,경복궁,O
1,삼청동 거리,O
2,인사동,O
3,조계사,O
4,청와대,O


In [5]:
df_x=df[df['일치여부']=='X']
df_x

,여행지,일치여부,다른 칸에 존재여부
6,이화장,X,X
11,대학로,X,X
24,정동길,X,X
36,이슬람사원,X,X
45,최순우 옛집,X,X
...,...,...,...
994,추사 거적지,X,X
995,정석항공관,X,X
997,제주 조각공원,X,X
1000,백조일손지묘,X,X


In [6]:
df_xx=df_x[df_x['다른 칸에 존재여부']!='X']
df_xx

,여행지,일치여부,다른 칸에 존재여부
51,월드컵공원,X,2
53,대한민국 국회의사당과 헌정기념관,X,3
66,암사동 선사주거지,X,2
69,화성,X,2
70,안양 예술공원,X,4
81,부천 활박물관과 궁도장,X,2
84,대부도,X,2
94,국립현대미술관(과천관),X,2
120,파주 프로방스,X,8
123,설봉공원,X,4


In [9]:
df_xx.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 55 entries, 51 to 991
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   여행지         55 non-null     object
 1   일치여부        55 non-null     object
 2   다른 칸에 존재여부  55 non-null     object
dtypes: object(3)
memory usage: 1.7+ KB


In [10]:
df_x[df_x['다른 칸에 존재여부']=='X'].count()

여행지           301
일치여부          301
다른 칸에 존재여부    301
dtype: int64

In [11]:
df_c

,여행지,일치여부
0,경복궁,O
1,삼청동 거리,O
2,인사동,O
3,조계사,O
4,청와대,O
...,...,...
996,알뜨르 비행장,O
998,이중섭 미술관,O
999,쇠소깍,O
1001,휴애리 자연생활공원,O


In [12]:
df_xx = df_xx.reset_index(drop = True)

In [13]:
df_xx = df_xx.astype({'다른 칸에 존재여부': 'int'})

In [23]:
df_xx.dtypes

여행지           object
일치여부          object
다른 칸에 존재여부     int32
dtype: object

In [14]:
df_xx

,여행지,일치여부,다른 칸에 존재여부
0,월드컵공원,X,2
1,대한민국 국회의사당과 헌정기념관,X,3
2,암사동 선사주거지,X,2
3,화성,X,2
4,안양 예술공원,X,4
5,부천 활박물관과 궁도장,X,2
6,대부도,X,2
7,국립현대미술관(과천관),X,2
8,파주 프로방스,X,8
9,설봉공원,X,4


2번째 : #BODY_BLOCK_JQUERY_REFLOW > div.page > div > div.ui_container.main_wrap > div > div > div > div > div.content_column.ui_column.is-9-desktop.is-12-tablet.is-12-mobile > div > div.ui_columns.sections_wrapper > div > div.prw_rup.prw_search_search_results.ajax-content > div > div.main_content.ui_column.is-12 > div > div:nth-child(2) > div > div > div:nth-child(2) > div > div > div > div.ui_column.is-9-desktop.is-8-mobile.is-9-tablet.content-block-column > div

3번째 : #BODY_BLOCK_JQUERY_REFLOW > div.page > div > div.ui_container.main_wrap > div > div > div > div > div.content_column.ui_column.is-9-desktop.is-12-tablet.is-12-mobile > div > div.ui_columns.sections_wrapper > div > div.prw_rup.prw_search_search_results.ajax-content > div > div.main_content.ui_column.is-12 > div > div:nth-child(2) > div > div > div:nth-child(3) > div > div > div > div.ui_column.is-9-desktop.is-8-mobile.is-9-tablet.content-block-column > div

 driver.find_element_by_css_selector('#BODY_BLOCK_JQUERY_REFLOW > div.page > div > div.ui_container.main_wrap > div > div > div > div > div.content_column.ui_column.is-9-desktop.is-12-tablet.is-12-mobile > div > div.ui_columns.sections_wrapper > div > div.prw_rup.prw_search_search_results.ajax-content > div > div.main_content.ui_column.is-12 > div > div:nth-child(2) > div > div > div:nth-child(%d) > div > div > div > div.ui_column.is-9-desktop.is-8-mobile.is-9-tablet.content-block-column > div').click()
 
 
 #BODY_BLOCK_JQUERY_REFLOW > div.page > div > div.ui_container.main_wrap > div > div > div > div > div.content_column.ui_column.is-9-desktop.is-12-tablet.is-12-mobile > div > div.ui_columns.sections_wrapper > div > div.prw_rup.prw_search_search_results.ajax-content > div > div.main_content.ui_column.is-12 > div > div:nth-child(2) > div > div > div:nth-child(2) > div > div > div > div.ui_column.is-9-desktop.is-8-mobile.is-9-tablet.content-block-column > div

In [7]:

options=webdriver.ChromeOptions()
options.add_argument('user_agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/101.0.4951.54 Safari/537.36')
options.add_experimental_option("excludeSwitches", ["enable-logging"])


driver=webdriver.Chrome()
url='https://www.tripadvisor.co.kr/'

reviews=[]
for i in range(len(df_c)):
    driver.get(url)
    search=df_c.여행지[i]
    driver.find_element_by_css_selector('#lithium-root > main > div.dqRmR.dcDXR.dJjeH > div > div > div.weiIG.Z0.Wh.fRhqZ > form > input.fhEMT._G.B-.z._J.Cj.R0').send_keys(search)
    driver.find_element_by_css_selector('#lithium-root > main > div.dqRmR.dcDXR.dJjeH > div > div > div.weiIG.Z0.Wh.fRhqZ > form > input.fhEMT._G.B-.z._J.Cj.R0').send_keys(Keys.ENTER)
    driver.switch_to.window(driver.window_handles[-1])
    time.sleep(2)
    try: 
        driver.find_element_by_css_selector('#BODY_BLOCK_JQUERY_REFLOW > div.page > div > div.ui_container.main_wrap > div > div > div > div > div.content_column.ui_column.is-9-desktop.is-12-tablet.is-12-mobile > div > div.ui_columns.sections_wrapper > div > div.prw_rup.prw_search_search_results.ajax-content > div > div.main_content.ui_column.is-12 > div > div:nth-child(2) > div > div > div:nth-child(1) > div > div > div > div.ui_column.is-9-desktop.is-8-mobile.is-9-tablet.content-block-column > div').click()

        driver.switch_to.window(driver.window_handles[-1])
        soup=BeautifulSoup(driver.page_source,'html.parser')
        mydata=soup.find_all('span','NejBf')
        for item in mydata:
            reviews.append(search + '-->' + item.get_text().strip())
        driver.switch_to.window(driver.window_handles[-1])
        driver.close()
        driver.switch_to.window(driver.window_handles[0])
        driver.back()
        time.sleep(1)
    except:
        pass

C:\Users\alsgu\AppData\Local\Temp/ipykernel_6764/3104573750.py:14: DeprecationWarning: find_element_by_* commands are deprecated. Please use find_element() instead
  driver.find_element_by_css_selector('#lithium-root > main > div.dqRmR.dcDXR.dJjeH > div > div > div.weiIG.Z0.Wh.fRhqZ > form > input.fhEMT._G.B-.z._J.Cj.R0').send_keys(search)
C:\Users\alsgu\AppData\Local\Temp/ipykernel_6764/3104573750.py:15: DeprecationWarning: find_element_by_* commands are deprecated. Please use find_element() instead
  driver.find_element_by_css_selector('#lithium-root > main > div.dqRmR.dcDXR.dJjeH > div > div > div.weiIG.Z0.Wh.fRhqZ > form > input.fhEMT._G.B-.z._J.Cj.R0').send_keys(Keys.ENTER)
C:\Users\alsgu\AppData\Local\Temp/ipykernel_6764/3104573750.py:20: DeprecationWarning: find_element_by_* commands are deprecated. Please use find_element() instead
  driver.find_element_by_css_selector('#BODY_BLOCK_JQUERY_REFLOW > div.page > div > div.ui_container.main_wrap > div > div > div > div > div.content

WebDriverException: Message: target frame detached
  (Session info: chrome=101.0.4951.67)
Stacktrace:
Backtrace:
	Ordinal0 [0x0052B8F3+2406643]
	Ordinal0 [0x004BAF31+1945393]
	Ordinal0 [0x003AC610+837136]
	Ordinal0 [0x0039D777+776055]
	Ordinal0 [0x0039C7C5+772037]
	Ordinal0 [0x0039CDC8+773576]
	Ordinal0 [0x0039CD58+773464]
	Ordinal0 [0x003A2F1A+798490]
	Ordinal0 [0x0039DF9B+778139]
	Ordinal0 [0x0039E4C5+779461]
	Ordinal0 [0x0039E2AF+778927]
	Ordinal0 [0x0039D8A6+776358]
	Ordinal0 [0x0039D810+776208]
	Ordinal0 [0x0039C7C5+772037]
	Ordinal0 [0x0039CDC8+773576]
	Ordinal0 [0x0039CD58+773464]
	Ordinal0 [0x003A2F1A+798490]
	Ordinal0 [0x0039DF9B+778139]
	Ordinal0 [0x0039E4C5+779461]
	Ordinal0 [0x0039E2AF+778927]
	Ordinal0 [0x0039D8A6+776358]
	Ordinal0 [0x0039D11C+774428]
	Ordinal0 [0x0039CFB9+774073]
	Ordinal0 [0x003ADB00+842496]
	Ordinal0 [0x00403F4B+1195851]
	Ordinal0 [0x003F4096+1130646]
	Ordinal0 [0x003CE636+976438]
	Ordinal0 [0x003CF546+980294]
	GetHandleVerifier [0x00799612+2498066]
	Ge

In [8]:
reviews

['경복궁-->대한민국의 역사',
 '경복궁-->대한민국의 역사가 잠들어 있는 곳. 서울을 방문했다면 꼭 방문해야 되는 곳. 경복궁은 우리의 역사다. 넓은 경복궁을 산책할 수 있는 것은 언제나 행복이다.',
 '경복궁-->국민이 공감하는 장소',
 '경복궁-->경복궁은 국민들이 자주 찾는곳으로 작성자는 주말에 자주 가족들과 방문하고 있음.특히 가족들과 방문시 옛고궁의 멋을 즐길수 있으며 쾌적한 공기와역사를 체험하고 힐링할수 있는곳임.',
 '경복궁-->산책하기 좋은 경복궁',
 '경복궁-->날씨 좋은 날 종종 산책하러 경복궁에 가는데 마음이 편온해지는 기분이라고 할까요? 특히 봄이나 가을에 산책가는 걸 추천해요!',
 '경복궁-->Good',
 '경복궁-->Goooooood 다 좋습니다 다음에 또 오고 싶네요 근처 관광지도많고 먹을거리도많네요 서울올때마다 자주이용중인데 또오고싶습니다',
 '경복궁-->가족단위로 방문하기 좋은곳',
 '경복궁-->요새 더더욱 코로나로 인해 사람 방문이 적음. 두자녀 동반시 성인 입장무료. 지금 시기에 나들이 하기 딱 좋은곳. 지하철 광화문역이나 종각역이용 편리.',
 '경복궁-->하늘이 내린 큰 복',
 "경복궁-->조선 개국 4년째인 1395년에 처음으로 세운 으뜸 궁궐이다.'하늘이 내린 큰 복' 이라는 뜻으로 경복궁이라 이름지었다.가장 아름답고 웅장한 이 경복궁은 임진왜란, 일제강점기 2번에 걸쳐 일본의 악랄하고 잔인한 파괴를 당한 악연을 갖고있다.향원정은 보수 공사로 인해 볼 수 없지만 아름다운 경회루만으로도 충분한 경복궁이다.",
 '경복궁-->굿',
 '경복궁-->한국 건축물의 아름다움을 느낄 수 있는 곳이고, 한복을 입었을 때 입장료는 무료.주변에 한복 대여점이 많으니 한번쯤 대여해서 입어보는것도 나쁘지 않을듯',
 '경복궁-->조선의 집',
 '경복궁-->조선의 성곽 안에는 박물관도 있고 경회루 등 볼것이 많다. 외국인들 및 연인들도 주변에서 한복 및 특수복장 빌려서 성을 둘러보며 데이트를 즐기기에 딱 좋다. 야간개

In [28]:
# 포맷팅 해서 적용하는 코드 (df_xx사용)
options=webdriver.ChromeOptions()
options.add_argument('user_agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/101.0.4951.54 Safari/537.36')
options.add_experimental_option("excludeSwitches", ["enable-logging"])


driver=webdriver.Chrome()
url='https://www.tripadvisor.co.kr/'


for i in range(len(df_xx)):
    driver.get(url)
    search=df_xx.여행지[i]
    driver.find_element_by_css_selector('#lithium-root > main > div.dqRmR.dcDXR.dJjeH > div > div > div.weiIG.Z0.Wh.fRhqZ > form > input.fhEMT._G.B-.z._J.Cj.R0').send_keys(search)
    driver.find_element_by_css_selector('#lithium-root > main > div.dqRmR.dcDXR.dJjeH > div > div > div.weiIG.Z0.Wh.fRhqZ > form > input.fhEMT._G.B-.z._J.Cj.R0').send_keys(Keys.ENTER)
    
    time.sleep(2)
    
    try : 
        if df['다른 칸에 존재여부']==2:
            driver.find_element_by_css_selector('#BODY_BLOCK_JQUERY_REFLOW > div.page > div > div.ui_container.main_wrap > div > div > div > div > div.content_column.ui_column.is-9-desktop.is-12-tablet.is-12-mobile > div > div.ui_columns.sections_wrapper > div > div.prw_rup.prw_search_search_results.ajax-content > div > div.main_content.ui_column.is-12 > div > div:nth-child(2) > div > div > div:nth-child(2) > div > div > div > div.ui_column.is-9-desktop.is-8-mobile.is-9-tablet.content-block-column > div').click()
            driver.switch_to.window(driver.window_handles[-1])
            soup=BeautifulSoup(driver.page_source,'html.parser')
            mydata=soup.find_all('span','NejBf')
            for item in mydata:
                reviews.append(search + '-->' + item.get_text().strip())
            driver.switch_to.window(driver.window_handles[-1])
            driver.close()
            driver.switch_to.window(driver.window_handles[0])
            driver.back()
            time.sleep(1)
        elif df['다른 칸에 존재여부']==3:
            driver.find_element_by_css_selector('#BODY_BLOCK_JQUERY_REFLOW > div.page > div > div.ui_container.main_wrap > div > div > div > div > div.content_column.ui_column.is-9-desktop.is-12-tablet.is-12-mobile > div > div.ui_columns.sections_wrapper > div > div.prw_rup.prw_search_search_results.ajax-content > div > div.main_content.ui_column.is-12 > div > div:nth-child(2) > div > div > div:nth-child(3) > div > div > div > div.ui_column.is-9-desktop.is-8-mobile.is-9-tablet.content-block-column > div').click()
            driver.switch_to.window(driver.window_handles[-1])
            soup=BeautifulSoup(driver.page_source,'html.parser')
            mydata=soup.find_all('span','NejBf')
            for item in mydata:
                reviews.append(search + '-->' + item.get_text().strip())
            driver.switch_to.window(driver.window_handles[-1])
            driver.close()
            driver.switch_to.window(driver.window_handles[0])
            driver.back()
            time.sleep(1)
        elif df['다른 칸에 존재여부']==4:
            driver.find_element_by_css_selector('#BODY_BLOCK_JQUERY_REFLOW > div.page > div > div.ui_container.main_wrap > div > div > div > div > div.content_column.ui_column.is-9-desktop.is-12-tablet.is-12-mobile > div > div.ui_columns.sections_wrapper > div > div.prw_rup.prw_search_search_results.ajax-content > div > div.main_content.ui_column.is-12 > div > div:nth-child(2) > div > div > div:nth-child(4) > div > div > div > div.ui_column.is-9-desktop.is-8-mobile.is-9-tablet.content-block-column > div').click()
            driver.switch_to.window(driver.window_handles[-1])
            soup=BeautifulSoup(driver.page_source,'html.parser')
            mydata=soup.find_all('span','NejBf')
            for item in mydata:
                reviews.append(search + '-->' + item.get_text().strip())
            driver.switch_to.window(driver.window_handles[-1])
            driver.close()
            driver.switch_to.window(driver.window_handles[0])
            driver.back()
            time.sleep(1)
        elif df['다른 칸에 존재여부']==5:
            driver.find_element_by_css_selector('#BODY_BLOCK_JQUERY_REFLOW > div.page > div > div.ui_container.main_wrap > div > div > div > div > div.content_column.ui_column.is-9-desktop.is-12-tablet.is-12-mobile > div > div.ui_columns.sections_wrapper > div > div.prw_rup.prw_search_search_results.ajax-content > div > div.main_content.ui_column.is-12 > div > div:nth-child(2) > div > div > div:nth-child(5) > div > div > div > div.ui_column.is-9-desktop.is-8-mobile.is-9-tablet.content-block-column > div').click()
            driver.switch_to.window(driver.window_handles[-1])
            soup=BeautifulSoup(driver.page_source,'html.parser')
            mydata=soup.find_all('span','NejBf')
            for item in mydata:
                reviews.append(search + '-->' + item.get_text().strip())
            driver.switch_to.window(driver.window_handles[-1])
            driver.close()
            driver.switch_to.window(driver.window_handles[0])
            driver.back()
            time.sleep(1)
        elif df['다른칸에 존재여부']==6:
            driver.find_element_by_css_selector('#BODY_BLOCK_JQUERY_REFLOW > div.page > div > div.ui_container.main_wrap > div > div > div > div > div.content_column.ui_column.is-9-desktop.is-12-tablet.is-12-mobile > div > div.ui_columns.sections_wrapper > div > div.prw_rup.prw_search_search_results.ajax-content > div > div.main_content.ui_column.is-12 > div > div:nth-child(2) > div > div > div:nth-child(6) > div > div > div > div.ui_column.is-9-desktop.is-8-mobile.is-9-tablet.content-block-column > div').click()
            driver.switch_to.window(driver.window_handles[-1])
            soup=BeautifulSoup(driver.page_source,'html.parser')
            mydata=soup.find_all('span','NejBf')
            for item in mydata:
                reviews.append(search + '-->' + item.get_text().strip())
            driver.switch_to.window(driver.window_handles[-1])
            driver.close()
            driver.switch_to.window(driver.window_handles[0])
            driver.back()
            time.sleep(1)
        elif df['다른 칸에 존재여부']==7:
            driver.find_element_by_css_selector('#BODY_BLOCK_JQUERY_REFLOW > div.page > div > div.ui_container.main_wrap > div > div > div > div > div.content_column.ui_column.is-9-desktop.is-12-tablet.is-12-mobile > div > div.ui_columns.sections_wrapper > div > div.prw_rup.prw_search_search_results.ajax-content > div > div.main_content.ui_column.is-12 > div > div:nth-child(2) > div > div > div:nth-child(7) > div > div > div > div.ui_column.is-9-desktop.is-8-mobile.is-9-tablet.content-block-column > div').click()
            driver.switch_to.window(driver.window_handles[-1])
            soup=BeautifulSoup(driver.page_source,'html.parser')
            mydata=soup.find_all('span','NejBf')
            for item in mydata:
                reviews.append(search + '-->' + item.get_text().strip())
            driver.switch_to.window(driver.window_handles[-1])
            driver.close()
            driver.switch_to.window(driver.window_handles[0])
            driver.back()
            time.sleep(1)
        elif df['다른 칸에 존재여부']==8:
            driver.find_element_by_css_selector('#BODY_BLOCK_JQUERY_REFLOW > div.page > div > div.ui_container.main_wrap > div > div > div > div > div.content_column.ui_column.is-9-desktop.is-12-tablet.is-12-mobile > div > div.ui_columns.sections_wrapper > div > div.prw_rup.prw_search_search_results.ajax-content > div > div.main_content.ui_column.is-12 > div > div:nth-child(2) > div > div > div:nth-child(8) > div > div > div > div.ui_column.is-9-desktop.is-8-mobile.is-9-tablet.content-block-column > div').click()
            driver.switch_to.window(driver.window_handles[-1])
            soup=BeautifulSoup(driver.page_source,'html.parser')
            mydata=soup.find_all('span','NejBf')
            for item in mydata:
                reviews.append(search + '-->' + item.get_text().strip())
            driver.switch_to.window(driver.window_handles[-1])
            driver.close()
            driver.switch_to.window(driver.window_handles[0])
            driver.back()
            time.sleep(1)
        elif df['다른 칸에 존재여부']==9:
            driver.find_element_by_css_selector('#BODY_BLOCK_JQUERY_REFLOW > div.page > div > div.ui_container.main_wrap > div > div > div > div > div.content_column.ui_column.is-9-desktop.is-12-tablet.is-12-mobile > div > div.ui_columns.sections_wrapper > div > div.prw_rup.prw_search_search_results.ajax-content > div > div.main_content.ui_column.is-12 > div > div:nth-child(2) > div > div > div:nth-child(9) > div > div > div > div.ui_column.is-9-desktop.is-8-mobile.is-9-tablet.content-block-column > div').click()
            driver.switch_to.window(driver.window_handles[-1])
            soup=BeautifulSoup(driver.page_source,'html.parser')
            mydata=soup.find_all('span','NejBf')
            for item in mydata:
                reviews.append(search + '-->' + item.get_text().strip())
            driver.switch_to.window(driver.window_handles[-1])
            driver.close()
            driver.switch_to.window(driver.window_handles[0])
            driver.back()
            time.sleep(1)
        elif df['다른 칸에 존재여부']==10:
            driver.find_element_by_css_selector('#BODY_BLOCK_JQUERY_REFLOW > div.page > div > div.ui_container.main_wrap > div > div > div > div > div.content_column.ui_column.is-9-desktop.is-12-tablet.is-12-mobile > div > div.ui_columns.sections_wrapper > div > div.prw_rup.prw_search_search_results.ajax-content > div > div.main_content.ui_column.is-12 > div > div:nth-child(2) > div > div > div:nth-child(10) > div > div > div > div.ui_column.is-9-desktop.is-8-mobile.is-9-tablet.content-block-column > div').click()
            driver.switch_to.window(driver.window_handles[-1])
            soup=BeautifulSoup(driver.page_source,'html.parser')
            mydata=soup.find_all('span','NejBf')
            for item in mydata:
                reviews.append(search + '-->' + item.get_text().strip())
            driver.switch_to.window(driver.window_handles[-1])
            driver.close()
            driver.switch_to.window(driver.window_handles[0])
            driver.back()
            time.sleep(1)
        else:
            pass
    except:
        pass

C:\Users\alsgu\AppData\Local\Temp/ipykernel_20220/3573077333.py:14: DeprecationWarning: find_element_by_* commands are deprecated. Please use find_element() instead
  driver.find_element_by_css_selector('#lithium-root > main > div.dqRmR.dcDXR.dJjeH > div > div > div.weiIG.Z0.Wh.fRhqZ > form > input.fhEMT._G.B-.z._J.Cj.R0').send_keys(search)
C:\Users\alsgu\AppData\Local\Temp/ipykernel_20220/3573077333.py:15: DeprecationWarning: find_element_by_* commands are deprecated. Please use find_element() instead
  driver.find_element_by_css_selector('#lithium-root > main > div.dqRmR.dcDXR.dJjeH > div > div > div.weiIG.Z0.Wh.fRhqZ > form > input.fhEMT._G.B-.z._J.Cj.R0').send_keys(Keys.ENTER)


NoSuchWindowException: Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: chrome=101.0.4951.54)
Stacktrace:
Backtrace:
	Ordinal0 [0x007E7413+2389011]
	Ordinal0 [0x00779F61+1941345]
	Ordinal0 [0x0066C658+837208]
	Ordinal0 [0x00654480+738432]
	Ordinal0 [0x006B6999+1141145]
	Ordinal0 [0x006C3AA2+1194658]
	Ordinal0 [0x006B3F66+1130342]
	Ordinal0 [0x0068E546+976198]
	Ordinal0 [0x0068F456+980054]
	GetHandleVerifier [0x00999632+1727522]
	GetHandleVerifier [0x00A4BA4D+2457661]
	GetHandleVerifier [0x0087EB81+569713]
	GetHandleVerifier [0x0087DD76+566118]
	Ordinal0 [0x00780B2B+1968939]
	Ordinal0 [0x00785988+1989000]
	Ordinal0 [0x00785A75+1989237]
	Ordinal0 [0x0078ECB1+2026673]
	BaseThreadInitThunk [0x75FBFA29+25]
	RtlGetAppContainerNamedObjectPath [0x77187A7E+286]
	RtlGetAppContainerNamedObjectPath [0x77187A4E+238]


In [23]:
reviews

['경복궁-->대한민국의 역사',
 '경복궁-->대한민국의 역사가 잠들어 있는 곳. 서울을 방문했다면 꼭 방문해야 되는 곳. 경복궁은 우리의 역사다. 넓은 경복궁을 산책할 수 있는 것은 언제나 행복이다.',
 '경복궁-->국민이 공감하는 장소',
 '경복궁-->경복궁은 국민들이 자주 찾는곳으로 작성자는 주말에 자주 가족들과 방문하고 있음.특히 가족들과 방문시 옛고궁의 멋을 즐길수 있으며 쾌적한 공기와역사를 체험하고 힐링할수 있는곳임.',
 '경복궁-->산책하기 좋은 경복궁',
 '경복궁-->날씨 좋은 날 종종 산책하러 경복궁에 가는데 마음이 편온해지는 기분이라고 할까요? 특히 봄이나 가을에 산책가는 걸 추천해요!',
 '경복궁-->Good',
 '경복궁-->Goooooood 다 좋습니다 다음에 또 오고 싶네요 근처 관광지도많고 먹을거리도많네요 서울올때마다 자주이용중인데 또오고싶습니다',
 '경복궁-->가족단위로 방문하기 좋은곳',
 '경복궁-->요새 더더욱 코로나로 인해 사람 방문이 적음. 두자녀 동반시 성인 입장무료. 지금 시기에 나들이 하기 딱 좋은곳. 지하철 광화문역이나 종각역이용 편리.',
 '경복궁-->하늘이 내린 큰 복',
 "경복궁-->조선 개국 4년째인 1395년에 처음으로 세운 으뜸 궁궐이다.'하늘이 내린 큰 복' 이라는 뜻으로 경복궁이라 이름지었다.가장 아름답고 웅장한 이 경복궁은 임진왜란, 일제강점기 2번에 걸쳐 일본의 악랄하고 잔인한 파괴를 당한 악연을 갖고있다.향원정은 보수 공사로 인해 볼 수 없지만 아름다운 경회루만으로도 충분한 경복궁이다.",
 '경복궁-->굿',
 '경복궁-->한국 건축물의 아름다움을 느낄 수 있는 곳이고, 한복을 입었을 때 입장료는 무료.주변에 한복 대여점이 많으니 한번쯤 대여해서 입어보는것도 나쁘지 않을듯',
 '경복궁-->조선의 집',
 '경복궁-->조선의 성곽 안에는 박물관도 있고 경회루 등 볼것이 많다. 외국인들 및 연인들도 주변에서 한복 및 특수복장 빌려서 성을 둘러보며 데이트를 즐기기에 딱 좋다. 야간개

In [ ]:
titles=[]
contents=[]
for i in range(0,len(reviews),2):
    titles.append(reviews[i])
for j in range(1,len(reviews),2):
    contents.append(reviews[j])

In [ ]:
data=pd.DataFrame({'리뷰제목':titles,'내용':contents})
data

In [ ]:
data.to_csv('여행지 리뷰 1차.csv',encoding='utf-8-sig',index=False)

In [ ]:
data['관광지']=data.리뷰제목.str.split('-->').str[0]
data['리뷰제목']=data.리뷰제목.str.split('-->').str[1]
data['내용']=data.내용.str.split('-->').str[1]

In [ ]:
data=data[['관광지','리뷰제목','내용']]
data

In [ ]:
data.to_csv('여행지 리뷰 2차.csv',encoding='utf-8-sig',index=False)

- 별도 실험용 코드

In [7]:
options=webdriver.ChromeOptions()
options.add_argument('user_agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/101.0.4951.54 Safari/537.36')
options.add_experimental_option("excludeSwitches", ["enable-logging"])

basepath='C:/Users/이신행/OneDrive/바탕 화면/chromedriver_win32(1)/'
driver=webdriver.Chrome()
url='https://www.tripadvisor.co.kr/Tourism-g294196-South_Korea-Vacations.html'

driver.get(url)
search=df.여행지[0]
driver.find_element_by_css_selector('#lithium-root > header > div > nav > div > div.fUFMI > div > div.weiIG.Z0.Wh.bZZEX > form > input.fhEMT._G.B-.z._J.Cj.R0').send_keys(search)
driver.find_element_by_css_selector('#lithium-root > header > div > nav > div > div.fUFMI > div > div.weiIG.Z0.Wh.bZZEX > form > input.fhEMT._G.B-.z._J.Cj.R0').send_keys(Keys.ENTER)
driver.switch_to.window(driver.window_handles[-1])
time.sleep(2)

soup=BeautifulSoup(driver.page_source,'html.parser')
mydata=soup.find_all('div','location-meta-block')
mydata[1].get_text().strip()

AttributeError: ResultSet object has no attribute 'get_text'. You're probably treating a list of items like a single item. Did you call find_all() when you meant to call find()?

In [8]:
mydata[1].get_text().strip()

'경복궁21건의 리뷰종로구 청진동 243, 서울, 대한민국종로구 청진동 243, 서울'